In [1]:
from stormpy import export_to_drn
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

from verimon.utils import setup_logging

setup_logging()

In [2]:
from verimon import loaders
from random import randrange
from math import sqrt

gb_e = "../tests/gb_exact.pm"
gb = loaders.load_dfa(gb_e)

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

def random_ladder(n):
    source = randrange(1,n-int(sqrt(n)))
    dest = randrange(source+int(sqrt(n)), int(min(n, source + n / 2)))
    return source, dest

def random_snake(n):
    source = randrange(int(sqrt(n))+2,n)
    dest = randrange(1, source-int(sqrt(n)))
    return source, dest

n = 5**2
ladders = {0:0}
snakes = {0:0}
while not set(ladders.keys()).isdisjoint(set(snakes.keys())):
    ladders = dict(random_ladder(n) for _ in range(int(sqrt(n))))
    snakes = dict(random_snake(n) for _ in range(int(sqrt(n))))

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [3]:

from stormvogel.mapping import stormpy_to_stormvogel, stormvogel_to_stormpy

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
# show(mc_sv)

In [4]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

player_path = [(0, [])]

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in mc_sv.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

<IPython.core.display.Javascript object>

In [5]:
from aalpy import run_Lstar
from verimon.MonitorLearning import FilteringSUL, VerimonEqOracle

setup_logging()


threshold = 0.4
slack = 0.1
horizon = 14
relative_error = 0.1
spec = 'P>0.5 [F<3 "good" ]'

alphabet = ["init", "normal", "snake", "ladder"]

filtering_sul = FilteringSUL(
    mc, 
    "init", 
    alphabet, 
    spec, 
    threshold, 
    horizon
)
eq_oracle = VerimonEqOracle(
    alphabet,
    filtering_sul,
    mc_sv,
    gb,
    threshold,
    slack,
    horizon,
    spec,
    'good',
    relative_error
    
)
learned_monitor = run_Lstar(
    alphabet,
    filtering_sul,
    eq_oracle,
    automaton_type="dfa",
    print_level=2,
)

Hypothesis 1: 1 states.
Visualization started in the background thread.
DEBUG:2024-10-23 16:16:04,060 - (0.00s) - MonitorLearning.py - Finding false negative probability 
DEBUG:2024-10-23 16:16:04,061 - (0.00s) - verify.py - Building model 
DEBUG:2024-10-23 16:16:04,063 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-10-23 16:16:04,068 - (0.00s) - verify.py - Pruning done 
INFO:2024-10-23 16:16:04,082 - (0.01s) - MDP_product.py - New good states become: [['[pos=80]'], ['[pos=100]']] 
DEBUG:2024-10-23 16:16:04,083 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-23 16:16:04,084 - (0.00s) - MDP_product.py - Finished product setup, created 14 observations 
DEBUG:2024-10-23 16:16:04,187 - (0.10s) - MDP_product.py - Created product 2830 states 
Model saved to model1.pdf.
DEBUG:2024-10-23 16:16:04,251 - (0.06s) - MDP_product.py - Created product transitions for 2828 states 
DEBUG:2024-10-23 16:16:04,252 - (0.00s) - verify.py - creating product done 
DEBUG:2024-10-23 16:16:04,253 - (

In [10]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.verify import *

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
export_to_drn(stormvogel_to_stormpy(mon_cycl), "monitor.drn")
result_goal, trace, assignment, product = false_positive(mc_sv, mon_cycl, gb, horizon, {"good_spec": spec})

Write to file monitor.drn.
DEBUG:2024-10-21 13:37:09,435 - (5388.11s) - verify.py - Building model 
DEBUG:2024-10-21 13:37:09,449 - (0.01s) - verify.py - Unrolling done 
DEBUG:2024-10-21 13:37:09,668 - (0.22s) - verify.py - Pruning done 
INFO:2024-10-21 13:37:09,684 - (0.02s) - MDP_product.py - New good states become: [['[pos=100]']] 
DEBUG:2024-10-21 13:37:09,685 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-21 13:37:09,686 - (0.00s) - MDP_product.py - Finished product setup, created 18 observations 
DEBUG:2024-10-21 13:37:49,006 - (39.32s) - MDP_product.py - Created product 50300 states 
DEBUG:2024-10-21 13:37:50,693 - (1.69s) - MDP_product.py - Created product transitions for 50298 states 
DEBUG:2024-10-21 13:37:50,699 - (0.01s) - verify.py - creating product done 
DEBUG:2024-10-21 13:37:50,699 - (0.00s) - verify.py - Creating trace 
ERROR (SpiritErrorHandler.h:26): Parsing error at 1:3:  expecting <expression>, here:
	P>{'good_spec': 'P>0.5 [F<3 "good" ]'} [F "stop"]
	  ^


RuntimeError: WrongFormatException: WrongFormatException: Parsing error at 1:3:  expecting <expression>, here:
	P>{'good_spec': 'P>0.5 [F<3 "good" ]'} [F "stop"]
	  ^
Note that the used API function does not have access to model variables. If the property you tried to parse contains model variables, it will not be parsed correctly.

In [9]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

# player_path = [(0, [])]
poss = product.simulate_paynt_assignment(assignment, 100000)
player_path = poss

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in product.mc.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

s0, labels=!s0 init !g0 [pos=0] !l0

	--[0, 3:ladder]-->
s609, labels=!g0 !l3 !s1 [pos=1] ladder
	--[1, 1:normal]-->
s1827, labels=!g0 !l9 !s7 [pos=38] normal
	--[2, 1:normal]-->
s3652, labels=!g0 !l18 !s14 [pos=39] normal
	--[3, 1:normal]-->
s6281, labels=!g0 !l31 !s17 [pos=42] normal
	--[4, 1:normal]-->
s9717, labels=!g0 !l48 !s19 [pos=44] normal
	--[5, 1:normal]-->
s13368, labels=!g0 !l66 !s34 [pos=50] normal
	--[6, 3:ladder]-->
s18027, labels=!g0 !l89 !s47 [pos=51] ladder
	--[7, 1:normal]-->
s23289, labels=!g0 !l115 !s57 [pos=67] normal
	--[8, 1:normal]-->
s28348, labels=!g0 !l140 !s66 [pos=69] normal
	--[9, 1:normal]-->
s34210, labels=!g0 !l169 !s70 [pos=73] normal
	--[11, 1:normal]-->
s40679, labels=!g0 !l201 !s75 [pos=75] normal
	--[13, 1:normal]-->
s46744, labels=!g0 !l231 !s80 [pos=79] normal
	--[15, 3:ladder]-->
s49169, labels=!g0 !l243 !s81 [pos=80] accepting horizon ladder
	--[17, 4:end]-->
s2, labels=stop
it took 3920 tries until the goal was reached


<IPython.core.display.Javascript object>